# 16 · Serve an agent-built web app and open it live (MicroVM)

An LLM writes a small web app, we run it **inside** a MicroVM, and open it in a
browser through a signed **preview link** — with an API key injected from
**secrets** and generated files kept on a **volume**.

**Key concepts**
- **`get_preview_link(port)`** — mints a short-lived, port-scoped
  `create_microvm_auth_token` and returns a `PreviewLink` (URL + `X-aws-proxy-*`
  headers) so an HTTP server *inside* the VM is reachable from outside.
- **`secrets={ENV: ref}`** — resolves `secretsmanager:<id>[::key]` / `ssm:<name>`
  / a literal via the AWS credential chain and injects it into the sandbox env,
  so the app reads a key it never sees in source.
- **`volume_mounts=[VolumeMount(fs_id, "/data")]`** — mounts an existing EFS file
  system so generated assets survive across sandboxes.
- All three need the **`microvm`** backend (a full OS + inbound network path).

**Prerequisites**
- AWS creds + region; a built image ARN in `RAITBOX_MICROVM_IMAGE`.
- Optional: `RAITBOX_EFS_ID` to exercise a real volume mount.
- Bedrock model access for `RAITBOX_BEDROCK_MODEL`.

In [ ]:
%pip install -q raitbox

### Configuration

In [ ]:
import os
import time

import boto3
import requests

from raitbox import Sandbox, VolumeMount

IMAGE_ARN = os.environ["RAITBOX_MICROVM_IMAGE"]
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("RAITBOX_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
EFS_ID = os.environ.get("RAITBOX_EFS_ID")  # optional
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

PORT = 8000

## 1 · Have the model write the app
We ask for a single-file FastAPI app that greets a name taken from the
environment (the value will arrive via `secrets`) and lists any files under
`/data` (the mounted volume).

In [ ]:
SPEC = f"""Write a single-file FastAPI app (app.py) that:
- reads GREETING_NAME from the environment (default "world")
- GET / returns JSON: {{"hello": GREETING_NAME, "saved_files": <names under /data>}}
- GET /save/{{name}} writes an empty file /data/{{name}} and returns {{"saved": name}}
Return ONLY the code, no fences, listen with uvicorn on 0.0.0.0:{PORT} under __main__."""

resp = bedrock.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": SPEC}]}],
    inferenceConfig={"maxTokens": 2048, "temperature": 0},
)
app_code = resp["output"]["message"]["content"][0]["text"].strip()
if app_code.startswith("```"):
    app_code = app_code.split("```")[1].removeprefix("python").strip()
print(app_code[:400], "...")

## 2 · Launch the VM with a secret and (optionally) a volume
`secrets` here uses a literal for a runnable demo; in production pass
`"secretsmanager:my/app/key"` or `"ssm:/app/token"` — same call, resolved from
AWS. `volume_mounts` is attached only when `RAITBOX_EFS_ID` is set.

In [ ]:
mounts = [VolumeMount(EFS_ID, "/data")] if EFS_ID else None

sbx = Sandbox.create(
    backend="microvm",
    template=IMAGE_ARN,
    secrets={"GREETING_NAME": "RaitBox"},  # ← or "secretsmanager:my/app/key"
    volume_mounts=mounts,
)
if not EFS_ID:
    sbx.files.make_dir("/data")  # stand in for the volume when none is mounted
print("sandbox:", sbx.sandbox_id)

## 3 · Run the app in the background and mint a preview link

In [ ]:
sbx.files.write("/app/app.py", app_code)
server = sbx.commands.run("cd /app && python app.py", background=True)

link = sbx.get_preview_link(PORT, expires_in_minutes=15)
print("preview URL:", link.url)
print("headers   :", link.headers)  # attach these to reach the port in the VM

## 4 · Call the app from here through the proxy
The preview URL + `link.headers` reach the server running inside the VM. A person
could paste the same into a browser extension that sets those headers.

In [ ]:
def call(path):
    for _ in range(15):  # wait for uvicorn to come up
        try:
            r = requests.get(link.url + path, headers=link.headers, timeout=5)
            if r.status_code < 500:
                return r.json()
        except requests.RequestException:
            pass
        time.sleep(1)
    raise RuntimeError("app did not respond")


print("root      :", call("/"))
print("save a.txt:", call("/save/a.txt"))
print("root again:", call("/"))  # a.txt now shows up (persisted on the volume)

In [ ]:
server.kill()
sbx.kill()
print("done")

## Recap
- `get_preview_link` turns a port in the VM into a signed, expiring URL.
- `secrets` injected the greeting without hard-coding it; swap the literal for a
  `secretsmanager:`/`ssm:` reference and nothing else changes.
- Files written under the mount persist on EFS across sandboxes.

**Related:** `12-background-and-snapshots` (get_host, background), `docs/CONCEPTS.md`.